# Prepare Qwen2.5 0.5B Instruct for TensorRT-LLM

Expected notebook path:
`qwen2.5-0.5b-instruct-trtllm/prepare_qwen2_5_0_5b_trtllm.ipynb`

This notebook downloads the Hugging Face model, converts it to a TensorRT-LLM checkpoint, builds a TensorRT-LLM engine, and writes a Triton model repository using `backend: "tensorrtllm"`.

It does **not** use `model.py`, `helpers.py`, or the Python LLMAPI backend.


In [ ]:
from pathlib import Path
import os
import subprocess
import shutil

# Run this notebook from the folder containing it.
EXAMPLE_DIR_NAME = "qwen2.5-0.5b-instruct-trtllm"
PROJECT = Path.cwd().resolve()
if PROJECT.name != EXAMPLE_DIR_NAME:
    raise RuntimeError(f"Run this notebook from {EXAMPLE_DIR_NAME}, got {PROJECT}")
if PROJECT.parent.name == EXAMPLE_DIR_NAME:
    raise RuntimeError(f"Nested example folder is wrong: {PROJECT}. Use one {EXAMPLE_DIR_NAME} folder only.")

MODEL_NAME = "qwen2_5_0_5b_instruct_trtllm"
HF_MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"

LOCAL_MODEL_DIR = PROJECT / "hf_model"
CKPT_DIR = PROJECT / "ckpt"
ENGINE_DIR = PROJECT / "engine"
TRITON_MODEL_DIR = PROJECT / MODEL_NAME
TRITON_VERSION_DIR = TRITON_MODEL_DIR / "1"
TRITON_TOKENIZER_DIR = TRITON_VERSION_DIR / "tokenizer"

# Avoid broken user-site packages and missing username/cache issues in arbitrary-UID containers.
os.environ["USER"] = "workspace"
os.environ["LOGNAME"] = "workspace"
os.environ["HOME"] = "/workspace"
os.environ["PYTHONNOUSERSITE"] = "1"

print("PROJECT:", PROJECT)
print("LOCAL_MODEL_DIR:", LOCAL_MODEL_DIR)
print("CKPT_DIR:", CKPT_DIR)
print("ENGINE_DIR:", ENGINE_DIR)
print("TRITON_MODEL_DIR:", TRITON_MODEL_DIR)
print("TRITON_VERSION_DIR:", TRITON_VERSION_DIR)
print("TRITON_TOKENIZER_DIR:", TRITON_TOKENIZER_DIR)


In [ ]:
# Find the Qwen checkpoint converter inside the TensorRT-LLM examples.
out = subprocess.check_output(
    "find / -path '*qwen*convert_checkpoint.py' 2>/dev/null | head -20",
    shell=True,
    text=True,
)
print(out)

paths = [p.strip() for p in out.splitlines() if p.strip()]
if not paths:
    raise RuntimeError("Could not find qwen convert_checkpoint.py")

CONVERT = paths[0]
print("CONVERT:", CONVERT)


In [ ]:
# Download the Hugging Face model into the project folder.
LOCAL_MODEL_DIR.mkdir(parents=True, exist_ok=True)

cmd = [
    "/opt/venv-tritonserver/bin/python",
    "-c",
    f'''
from huggingface_hub import snapshot_download
snapshot_download(
    repo_id="{HF_MODEL_ID}",
    local_dir="{LOCAL_MODEL_DIR}",
    local_dir_use_symlinks=False
)
print("{LOCAL_MODEL_DIR}")
'''
]

env = dict(os.environ)
env["PYTHONNOUSERSITE"] = "1"
env.pop("PYTHONPATH", None)

result = subprocess.run(cmd, text=True, capture_output=True, env=env)
print("STDOUT:\n", result.stdout)
print("STDERR:\n", result.stderr)

if result.returncode != 0:
    raise RuntimeError("Hugging Face model download failed")


In [ ]:
# Convert the local Hugging Face model to a TensorRT-LLM checkpoint.
if CKPT_DIR.exists():
    shutil.rmtree(CKPT_DIR)

cmd = [
    "/opt/venv-tritonserver/bin/python",
    CONVERT,
    "--model_dir", str(LOCAL_MODEL_DIR),
    "--output_dir", str(CKPT_DIR),
    "--dtype", "float16",
    "--tp_size", "1",
]

env = dict(os.environ)
env["PYTHONNOUSERSITE"] = "1"
env.pop("PYTHONPATH", None)

print("Running:")
print(" ".join(cmd))

result = subprocess.run(cmd, cwd=str(PROJECT), text=True, capture_output=True, env=env)
print("STDOUT:\n", result.stdout)
print("STDERR:\n", result.stderr)

if result.returncode != 0:
    raise RuntimeError(f"convert_checkpoint.py failed with exit code {result.returncode}")

print("Checkpoint files:")
for p in CKPT_DIR.rglob("*"):
    print(p)


In [ ]:
# Build the TensorRT-LLM engine.
if ENGINE_DIR.exists():
    shutil.rmtree(ENGINE_DIR)

cmd = [
    "trtllm-build",
    "--checkpoint_dir", str(CKPT_DIR),
    "--output_dir", str(ENGINE_DIR),
    "--gemm_plugin", "float16",
    "--max_batch_size", "1",
    "--max_input_len", "1024",
    "--max_seq_len", "2048",
]

env = dict(os.environ)
env["PYTHONNOUSERSITE"] = "1"
env.pop("PYTHONPATH", None)

print("Running:")
print(" ".join(cmd))

result = subprocess.run(cmd, cwd=str(PROJECT), text=True, capture_output=True, env=env)
print("STDOUT:\n", result.stdout)
print("STDERR:\n", result.stderr)

if result.returncode != 0:
    raise RuntimeError(f"trtllm-build failed with exit code {result.returncode}")

engine_file = ENGINE_DIR / "rank0.engine"
if not engine_file.exists():
    raise RuntimeError(f"trtllm-build finished, but did not create {engine_file}")

print("Engine files:")
for p in ENGINE_DIR.rglob("*"):
    print(p)


In [ ]:
# Create the Triton model directory and copy the engine files into version folder 1/.
# If engine/rank0.engine is missing, build it from ckpt/ first.
engine_file = ENGINE_DIR / "rank0.engine"
if not engine_file.exists():
    ckpt_config = CKPT_DIR / "config.json"
    if not ckpt_config.exists():
        raise RuntimeError(f"Missing {ckpt_config}. Run the checkpoint conversion cell first.")

    if ENGINE_DIR.exists():
        shutil.rmtree(ENGINE_DIR)

    cmd = [
        "trtllm-build",
        "--checkpoint_dir", str(CKPT_DIR),
        "--output_dir", str(ENGINE_DIR),
        "--gemm_plugin", "float16",
        "--max_batch_size", "1",
        "--max_input_len", "1024",
        "--max_seq_len", "2048",
    ]

    env = dict(os.environ)
    env["PYTHONNOUSERSITE"] = "1"
    env.pop("PYTHONPATH", None)

    print(f"Missing {engine_file}; building engine now.")
    print("Running:")
    print(" ".join(cmd))

    result = subprocess.run(cmd, cwd=str(PROJECT), text=True, capture_output=True, env=env)
    print("STDOUT:\n", result.stdout)
    print("STDERR:\n", result.stderr)

    if result.returncode != 0:
        raise RuntimeError(f"trtllm-build failed with exit code {result.returncode}")

if not engine_file.exists():
    raise RuntimeError(f"Missing {engine_file}; trtllm-build did not produce rank0.engine.")

if TRITON_MODEL_DIR.exists():
    shutil.rmtree(TRITON_MODEL_DIR)

TRITON_VERSION_DIR.mkdir(parents=True, exist_ok=True)

for item in ENGINE_DIR.iterdir():
    dest = TRITON_VERSION_DIR / item.name
    if item.is_dir():
        shutil.copytree(item, dest)
    else:
        shutil.copy2(item, dest)

TRITON_TOKENIZER_DIR.mkdir(parents=True, exist_ok=True)
for pattern in ["tokenizer*", "special_tokens_map.json", "added_tokens.json", "vocab.json", "merges.txt", "config.json", "generation_config.json"]:
    for src in LOCAL_MODEL_DIR.glob(pattern):
        if src.is_file():
            shutil.copy2(src, TRITON_TOKENIZER_DIR / src.name)

print("Copied engine to:", TRITON_VERSION_DIR)
print("Copied tokenizer files to:", TRITON_TOKENIZER_DIR)
for p in TRITON_VERSION_DIR.rglob("*"):
    print(p)


In [ ]:
# Write config.pbtxt for the TensorRT-LLM Triton backend.
config = f'''
name: "{MODEL_NAME}"
backend: "tensorrtllm"
max_batch_size: 1

model_transaction_policy {{
  decoupled: true
}}

input [
  {{
    name: "input_ids"
    data_type: TYPE_INT32
    dims: [ -1 ]
  }},
  {{
    name: "input_lengths"
    data_type: TYPE_INT32
    dims: [ 1 ]
  }},
  {{
    name: "request_output_len"
    data_type: TYPE_INT32
    dims: [ 1 ]
  }}
]

output [
  {{
    name: "output_ids"
    data_type: TYPE_INT32
    dims: [ -1, -1 ]
  }},
  {{
    name: "sequence_length"
    data_type: TYPE_INT32
    dims: [ -1 ]
  }}
]

parameters: {{
  key: "triton_backend"
  value: {{ string_value: "tensorrtllm" }}
}}

parameters: {{
  key: "triton_max_batch_size"
  value: {{ string_value: "1" }}
}}

parameters: {{
  key: "decoupled_mode"
  value: {{ string_value: "true" }}
}}

parameters: {{
  key: "engine_dir"
  value: {{ string_value: "/models/{MODEL_NAME}/1" }}
}}

parameters: {{
  key: "gpt_model_type"
  value: {{ string_value: "inflight_fused_batching" }}
}}

parameters: {{
  key: "gpt_model_path"
  value: {{ string_value: "/models/{MODEL_NAME}/1" }}
}}

parameters: {{
  key: "batching_strategy"
  value: {{ string_value: "inflight_fused_batching" }}
}}

parameters: {{
  key: "batch_scheduler_policy"
  value: {{ string_value: "guaranteed_no_evict" }}
}}

parameters: {{
  key: "max_queue_delay_microseconds"
  value: {{ string_value: "0" }}
}}

parameters: {{
  key: "max_queue_size"
  value: {{ string_value: "0" }}
}}

parameters: {{
  key: "max_beam_width"
  value: {{ string_value: "1" }}
}}

parameters: {{
  key: "kv_cache_free_gpu_mem_fraction"
  value: {{ string_value: "0.85" }}
}}

parameters: {{
  key: "exclude_input_in_output"
  value: {{ string_value: "true" }}
}}

parameters: {{
  key: "enable_kv_cache_reuse"
  value: {{ string_value: "false" }}
}}

parameters: {{
  key: "encoder_input_features_data_type"
  value: {{ string_value: "TYPE_FP16" }}
}}

parameters: {{
  key: "logits_datatype"
  value: {{ string_value: "TYPE_FP32" }}
}}

parameters: {{
  key: "prompt_embedding_table_data_type"
  value: {{ string_value: "TYPE_FP16" }}
}}

parameters: {{
  key: "guided_decoding_backend"
  value: {{ string_value: "" }}
}}

parameters: {{
  key: "tokenizer_dir"
  value: {{ string_value: "1/tokenizer" }}
}}

parameters: {{
  key: "xgrammar_tokenizer_info_path"
  value: {{ string_value: "" }}
}}
'''

config_path = TRITON_MODEL_DIR / "config.pbtxt"
config_path.write_text(config)

print("WROTE:", config_path)
print(config_path.read_text())

# Remove temporary build folders. The Triton model repository now contains the engine and tokenizer.
for temp_dir in [LOCAL_MODEL_DIR, CKPT_DIR, ENGINE_DIR]:
    if temp_dir.exists():
        shutil.rmtree(temp_dir)
        print("Removed:", temp_dir)


In [ ]:
# Final check. There should be config.pbtxt and engine files under 1/.
print("Final Triton model files:")
for p in TRITON_MODEL_DIR.rglob("*"):
    print(p)

print("\nImportant: this model repository should not contain model.py or helpers.py.")
